In [ ]:
import copernicusmarine
import xarray as xr

In [ ]:
ds = copernicusmarine.open_dataset(
  dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",)
ds

In [ ]:
ssv = ds.sel(longitude=slice(-30,-16), latitude=slice(12,20)).sel(depth=0, method="nearest")[["uo", "vo"]]
ssv

In [ ]:
ssv.to_netcdf("ssv.nc")

In [ ]:
flowfield_env = xr.open_dataset("../data/ssc.nc").squeeze()
flowfield_env

In [ ]:
ds_chl = copernicusmarine.open_dataset(
  dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D",)
ds_chl

In [ ]:
ds_chl = ds_chl.sel(longitude=slice(-29,-20), latitude=slice(13,20))[["CHL"]]
ds_chl

In [ ]:
ds_chl.to_netcdf("../data/chl.nc")

In [13]:
ds_ssh = copernicusmarine.open_dataset(
  dataset_id="cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D",)
ds_ssh

INFO - 2026-07-28T20:21:06Z - Selected dataset version: "202411"
INFO - 2026-07-28T20:21:06Z - Selected dataset part: "default"


<xarray.Dataset> Size: 4TB
Dimensions:         (time: 12069, latitude: 1440, longitude: 2880)
Coordinates:
  * latitude        (latitude) float32 6kB -89.94 -89.81 -89.69 ... 89.81 89.94
  * longitude       (longitude) float32 12kB -179.9 -179.8 ... 179.8 179.9
  * time            (time) datetime64[ns] 97kB 1993-01-01 ... 2026-01-16
Data variables:
    adt             (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    err_sla         (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    err_ugosa       (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    err_vgosa       (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    flag_ice        (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    sla             (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    tpa_correction  (time) float64 97kB dask.array<chunksize=(50,), meta=np.ndarray>
    ugos            (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    ugosa           (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    vgos            (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
    vgosa           (time, latitude, longitude) float64 400GB dask.array<chunksize=(50, 1024, 1024), meta=np.ndarray>
Attributes: (12/43)
    Conventions:                     CF-1.6
    Metadata_Conventions:            Unidata Dataset Discovery v1.0
    cdm_data_type:                   Grid
    comment:                         Sea Surface Height measured by Altimetry...
    contact:                         servicedesk.cmems@mercator-ocean.eu
    creator_email:                   servicedesk.cmems@mercator-ocean.eu
    ...                              ...
    time_coverage_duration:          P1D
    time_coverage_end:               2023-12-31T12:00:00Z
    time_coverage_resolution:        P1D
    time_coverage_start:             2023-12-30T12:00:00Z
    title:                           DT merged all satellites Global Ocean Gr...
    copernicusmarine_version:        2.4.1

In [14]:
ds_ssh = ds_ssh.sel(longitude=slice(-30,-16), latitude=slice(12,20))["adt"]
ds_ssh

<xarray.DataArray 'adt' (time: 12069, latitude: 64, longitude: 112)> Size: 692MB
dask.array<getitem, shape=(12069, 64, 112), dtype=float64, chunksize=(50, 64, 112), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float32 256B 12.06 12.19 12.31 ... 19.69 19.81 19.94
  * longitude  (longitude) float32 448B -29.94 -29.81 -29.69 ... -16.19 -16.06
  * time       (time) datetime64[ns] 97kB 1993-01-01 1993-01-02 ... 2026-01-16
Attributes:
    comment:        The absolute dynamic topography is the sea surface height...
    grid_mapping:   crs
    long_name:      Absolute dynamic topography
    standard_name:  sea_surface_height_above_geoid
    units:          m

In [15]:
ds_ssh.to_netcdf("../data/ssh_data.nc")

In [2]:
from glob import glob
import xarray as xr

path = "/home/b/b384140/work_bk1450/lagrangian-flood-analysis-thesis/data/hourly_glorys"

# Open all January files
gridT = xr.open_mfdataset(
    sorted(glob(f"{path}/gridT_cabo_verde_2020-01-*.nc")),
    combine="by_coords"
)

gridU = xr.open_mfdataset(
    sorted(glob(f"{path}/gridU_cabo_verde_2020-01-*.nc")),
    combine="by_coords"
)

gridV = xr.open_mfdataset(
    sorted(glob(f"{path}/gridV_cabo_verde_2020-01-*.nc")),
    combine="by_coords"
)

# Merge variables into one Dataset
ds = xr.merge([gridT, gridU, gridV], compat="override")

# Save
ds.to_netcdf(
    f"{path}/glorys_cabo_verde_2020-01.nc",
    format="NETCDF4"
)

In [3]:
import os
import time
import glob
import threading

import requests
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

from pydap.client import open_url
from xarray.backends import PydapDataStore
import pydap.net as pydap_net
import pydap.handlers.dap as dap_module

from concurrent.futures import ThreadPoolExecutor, as_completed

In [5]:
# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
CA_BUNDLE = "/home/b/b384140/combined_ca_bundle.pem"
AUTHELIA_COOKIE = "zwOB%VudzCnylviGY5CcGdw%FOCE$aWh"  # refresh if it's been a while

LON_MIN, LON_MAX = -30,-16
LAT_MIN, LAT_MAX = 12,20

START_DATE = "2020-01-01"
END_DATE   = "2025-12-31"

OUT_DIR = "../data/hourly_glorys"
os.makedirs(OUT_DIR, exist_ok=True)

BASE = "https://tds.mercator-ocean.fr/thredds/dodsC"

# Confirmed via catalog crawl (see 002_download_hourly.ipynb Step 1 pattern):
#   MOI GLORYS12 [GLORYS12V1] > [PGN Hourly]
#     glorys12v1-hourly-gridT / gridU / gridV
DATASETS = {
    "gridT": f"{BASE}/glorys12v1-hourly-gridT",
    "gridU": f"{BASE}/glorys12v1-hourly-gridU",
    "gridV": f"{BASE}/glorys12v1-hourly-gridV",
}

GRID_VARS_HINT = {
    "gridT": None,
    "gridU": ["vozocrtx", "uo", "sozocrtx"],
    "gridV": ["vomecrty", "vo", "somecrty"],
}

# Same static files you already used from the PSY4V3R1 side, re-exposed
# under the glorys12v1 catalog folder (identical content):
MASK_URL     = f"{BASE}/glorys12v1_statics/PSY4V3R1_mask.nc"
MESH_HGR_URL = f"{BASE}/glorys12v1_statics/PSY4V3R1_mesh_hgr.nc"

MAX_WORKERS = 12  # keep conservative -- hourly requests are heavier per file

In [6]:
pdap_mesh = open_url(MESH_HGR_URL, session=mesh_session, protocol="dap2", checksums=False)
mesh = xr.open_dataset(PydapDataStore(pdap_mesh))

mesh_sub = mesh.isel(y=j_slice, x=i_slice)
if "t" in mesh_sub.dims:
    mesh_sub = mesh_sub.isel(t=0)   # mesh vars carry a singleton time dim -- squeeze it

lon_t_mesh, lat_t_mesh = mesh_sub["glamt"].values, mesh_sub["gphit"].values  # T-point (centers)
lon_f_mesh, lat_f_mesh = mesh_sub["glamf"].values, mesh_sub["gphif"].values  # F-point (corners)

print("lon_f shape:", lon_f_mesh.shape, "-- should match (len(j_slice), len(i_slice))")

# sanity check: T-point mesh coords should match the T-grid coords you
# actually downloaded
ds_check = xr.open_dataset("../data/hourly_glorys/glorys_cabo_verde_2020-01.nc")
print("max |lon_t mesh - lon_t data| =", np.nanmax(np.abs(lon_t_mesh - ds_check["lon_t"].values)))
print("max |lat_t mesh - lat_t data| =", np.nanmax(np.abs(lat_t_mesh - ds_check["lat_t"].values)))

NameError: name 'mesh_session' is not defined